# Inverse trigonometric functions

This notebook covers `es.math.arcsin`, `arccos`, `arctan` and `arctan2`. They take a `Parameter` or a
`DescriptorNumber` and return an **angle**, in radians by default; the keyword `unit='deg'` returns degrees,
with the variance and the bounds converted to match. The variance is propagated to first order, and the
bounds are the exact range of the function over the bounds of the input.

The examples are taken from a diffraction experiment, but each one stands alone. Each part closes with a few exercises on what was just shown. Their hints and solutions are folded away and open on a click, so the text reads straight through; a solution is a worked answer with its result.

In [ ]:
import numpy as np
import plotly.graph_objects as go

import easyscience as es
from easyscience import DescriptorNumber
from easyscience import Parameter

The function `show` prints a number as value, error and unit, and, for a `Parameter` with finite bounds, the bounds.

In [ ]:
def show(number):
    """One line with value, error, unit and, for a Parameter, the finite bounds."""
    text = f'{number.value:.4f} ± {number.error or 0.0:.4f} {number.unit}'
    if hasattr(number, 'min') and (np.isfinite(number.min) or np.isfinite(number.max)):
        text += f'   in [{number.min:.4f}, {number.max:.4f}]'
    return text

In [ ]:
import base64
import contextlib
import html
import io
import re

from IPython import get_ipython
from IPython.display import HTML
from IPython.display import IFrame
from IPython.display import display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))


def solution(note: str, source: str) -> None:
    """Run a solution in a copy of the notebook namespace and show it, folded, with its output."""
    code = source.strip('\n')
    namespace = dict(get_ipython().user_ns)
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        exec(code, namespace)
    parts = [
        '<details class="example"><summary>Solution</summary>',
        highlight(code, PythonLexer(), HtmlFormatter()),
    ]
    result = stream.getvalue().rstrip('\n')
    if result:
        parts.append('<p>Output:</p><pre>' + html.escape(result) + '</pre>')
    if note.strip():
        text = re.sub('`([^`]+)`', r'<code>\1</code>', html.escape(note.strip()))
        parts.append('<p>' + text + '</p>')
    parts.append('</details>')
    display(HTML(''.join(parts)))


get_ipython().register_magic_function(solution, magic_kind='cell')

## The arcsine and the Bragg angle

Bragg's law $\lambda = 2d\sin\theta$ gives the scattering angle from the lattice spacing $d$ and the
wavelength $\lambda$, with $\theta = \arcsin\!\left(\lambda/(2d)\right)$. The argument is dimensionless
because the two lengths cancel:

In [ ]:
d = Parameter(name='d', value=3.1356, unit='angstrom', variance=0.05**2, min=2.0, max=6.0)
wavelength = DescriptorNumber(name='wavelength', value=1.5406, unit='angstrom', variance=0.0)

print(show(es.math.arcsin(wavelength / (2 * d))))
print(show(es.math.arcsin(wavelength / (2 * d), unit='deg')))

The result is in radians unless `unit='deg'` says otherwise. A larger $d$ gives a smaller angle, so the
upper bound of the angle comes from the lower bound of $d$.

### Exercise 1: The Bragg angle of silicon

The (111) planes of silicon have a spacing of $3.1356 \pm 0.0005$ Å. For X-rays of wavelength 1.5406 Å (known exactly), what is the Bragg angle $\theta$, in degrees?

In [ ]:
d = Parameter(name='d', value=3.1356, variance=0.0005**2, unit='angstrom')
wavelength = DescriptorNumber(name='wavelength', value=1.5406, variance=0.0, unit='angstrom')

<details class="question">
<summary>Hint 1</summary>

$\theta = \arcsin\!\left(\lambda/(2d)\right)$; form the ratio $\lambda/(2d)$ first.

</details>

<details class="question">
<summary>Hint 2</summary>

Ask for degrees with the keyword `unit='deg'`.

</details>

In [ ]:
%%solution The bounds of the arcsine are $\pm 90^\circ$ for an unbounded input.
theta = es.math.arcsin(wavelength / (2 * d), unit='deg')
print(show(theta))

## The arccosine and the arctangent

`arccos` returns an angle between $0$ and $\pi$, and `arctan` an angle between $-\pi/2$ and $\pi/2$.
The arccosine of a number between 0 and 1:

In [ ]:
ratio = Parameter(name='ratio', value=0.5, variance=0.02**2, min=0.0, max=1.0)
print(show(es.math.arccos(ratio)))

The arctangent gives the angle of a slope. A ramp that rises 10 cm over a length of 1 m has a grade of 0.10:

In [ ]:
grade = Parameter(name='grade', value=0.10, variance=0.005**2)
print(show(es.math.arctan(grade, unit='deg')))

### Exercise 2: Angle of a slope

A ramp rises $0.30 \pm 0.01$ m over a horizontal run of $1.20 \pm 0.02$ m. What is its angle with the horizontal, in degrees?

In [ ]:
rise = Parameter(name='rise', value=0.30, variance=0.01**2, unit='m')
run = Parameter(name='run', value=1.20, variance=0.02**2, unit='m')

<details class="question">
<summary>Hint 1</summary>

The tangent of the angle is the rise divided by the run.

</details>

<details class="question">
<summary>Hint 2</summary>

The ratio of two lengths in the same unit is dimensionless, which is what `arctan` needs.

</details>

In [ ]:
%%solution
angle = es.math.arctan(rise / run, unit='deg')
print(show(angle))

## The two-argument arctangent

`arctan(b / a)` cannot tell $(a, b)$ from $(-a, -b)$. `arctan2(b, a)` uses the signs of both arguments to
find the right quadrant. The phase of a structure factor $F = a + \mathrm{i}\,b$ is `arctan2(b, a)`; the two
arguments must have the same unit (here none):

In [ ]:
a = Parameter(name='a', value=3.0, variance=0.1**2, min=2.0, max=4.0)
b = Parameter(name='b', value=4.0, variance=0.1**2, min=3.0, max=5.0)
a_negative = Parameter(name='a_negative', value=-3.0, variance=0.1**2, min=-4.0, max=-2.0)

print(show(es.math.arctan2(b, a, unit='deg')))
print(show(es.math.arctan2(b, a_negative, unit='deg')))
print(show(es.math.arctan(b / a_negative, unit='deg')))

The second line is the angle in the second quadrant, $126.87^\circ$. The third line, with plain `arctan`,
cannot know about the quadrant and gives $-53.13^\circ$.

The shorter names `asin`, `acos`, `atan` and `atan2` are aliases of the same functions:

In [ ]:
es.math.asin is es.math.arcsin

### Exercise 3: Phase of a structure factor

The real and imaginary parts of a structure factor are $a = 1.5 \pm 0.05$ and $b = 2.0 \pm 0.05$. What is its phase, in degrees?

In [ ]:
a = Parameter(name='a', value=1.5, variance=0.05**2)
b = Parameter(name='b', value=2.0, variance=0.05**2)

<details class="question">
<summary>Hint 1</summary>

The phase is `arctan2` of the imaginary part over the real part: the **first** argument is `b`.

</details>

<details class="question">
<summary>Hint 2</summary>

Use the keyword `unit='deg'` for degrees.

</details>

In [ ]:
%%solution The bounds are $\pm 180^\circ$, the full range of the function.
phase = es.math.arctan2(b, a, unit='deg')
print(show(phase))

## Values outside the domain

`arcsin` and `arccos` accept numbers between -1 and 1 only. A ratio $\lambda/(2d)$ larger than 1 means that
**no angle satisfies Bragg's law** for that wavelength and spacing, and the function says so instead of
returning a meaningless number. The error message names the parameter it was given.

### Exercise 4: No Bragg angle

A crystal has a lattice spacing of $0.70 \pm 0.01$ Å, and the X-rays have a wavelength of 1.5406 Å. Predict what `es.math.arcsin(wavelength / (2 * d), unit='deg')` does, then try it. Catch the error and print its message.

In [ ]:
d = Parameter(name='d', value=0.70, variance=0.01**2, unit='angstrom')
wavelength = DescriptorNumber(name='wavelength', value=1.5406, variance=0.0, unit='angstrom')

<details class="question">
<summary>Hint 1</summary>

Look at the value of `wavelength / (2 * d)`: which numbers does `arcsin` accept?

</details>

<details class="question">
<summary>Hint 2</summary>

Put the call in a `try` block and catch `ValueError`. Give the intermediate result a readable name (`ratio.name = 'ratio'`), because the error message names the parameter it was given.

</details>

In [ ]:
%%solution The ratio is larger than 1, so no angle exists. Without the line that sets the name, the message would name an automatically generated parameter instead of `ratio`.
ratio = wavelength / (2 * d)
ratio.name = 'ratio'
try:
    es.math.arcsin(ratio, unit='deg')
except ValueError as error:
    print(error)

## A round trip

The sine and the arcsine undo each other. The Bragg angle recovered from $d$ and $\lambda$ is the angle we
started with, and its error is almost the same:

In [ ]:
theta = Parameter(name='theta', value=15.0, unit='deg', variance=0.1**2, min=5.0, max=60.0)
wavelength = DescriptorNumber(name='wavelength', value=1.5406, unit='angstrom', variance=0.0005**2)

d = wavelength / (2 * es.math.sin(theta))
recovered = es.math.arcsin(wavelength / (2 * d), unit='deg')
print(show(recovered))
print(f'variance after the round trip: {recovered.variance:.6f}')
print(f'variance of theta:             {theta.variance:.6f}')

The excess is small and has a simple cause. The wavelength cancels mathematically, so the variance should
come back as exactly that of `theta`. But `d` is a stored intermediate result, so the second use of the
wavelength is treated as a new, independent number, just as in $(2x)/x$ in the notebook on arithmetic.

## Charts

The Bragg angle as a function of the wavelength, for a crystal with $d = 3.1356$ Å and a wavelength with an
uncertainty of 0.01 Å. The band widens as $\lambda/(2d)$ approaches 1: the derivative of the arcsine
diverges there. The chart is interactive: hover over the curve to read the values. It loads Plotly from the
internet, so it needs a connection. The charts are shown with a small helper, `show_figure(fig)`, which is
defined in a hidden cell; in your own notebook `fig.show()` does the same.

In [ ]:
wavelengths = np.linspace(0.5, 6.0, 56)
spacing = DescriptorNumber(name='d', value=3.1356, unit='angstrom', variance=0.0)

values, errors = [], []
for value in wavelengths:
    wavelength = Parameter(name='wavelength', value=value, unit='angstrom', variance=0.01**2)
    angle = es.math.arcsin(wavelength / (2 * spacing), unit='deg')
    values.append(angle.value)
    errors.append(angle.error)
values, errors = np.array(values), np.array(errors)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=wavelengths,
        y=values - errors,
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=wavelengths,
        y=values + errors,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(99, 110, 250, 0.25)',
        name='± 1 error',
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=wavelengths,
        y=values,
        mode='lines',
        name='Bragg angle',
        line=dict(color='rgb(99, 110, 250)'),
        customdata=errors,
        hovertemplate='wavelength = %{x:.2f} Å<br>theta = %{y:.2f} ± %{customdata:.3f} deg<extra></extra>',
    )
)
fig.update_layout(
    xaxis_title='wavelength / Å',
    yaxis_title='Bragg angle theta / deg',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.05, y=0.95),
)
show_figure(fig)

The phase of a structure factor $a + \mathrm{i}\,b$ with $a = 1.5 \pm 0.05$ as the imaginary part $b$ goes
from $-3$ to $3$ (with an uncertainty of 0.05). The phase runs from $-63^\circ$ to $63^\circ$ and is steepest
near $b = 0$, where the error is largest:

In [ ]:
imaginary = np.linspace(-3.0, 3.0, 61)
real = Parameter(name='a', value=1.5, variance=0.05**2)

values, errors = [], []
for value in imaginary:
    b = Parameter(name='b', value=value, variance=0.05**2)
    phase = es.math.arctan2(b, real, unit='deg')
    values.append(phase.value)
    errors.append(phase.error)
values, errors = np.array(values), np.array(errors)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=imaginary,
        y=values - errors,
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=imaginary,
        y=values + errors,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(99, 110, 250, 0.25)',
        name='± 1 error',
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=imaginary,
        y=values,
        mode='lines',
        name='phase',
        line=dict(color='rgb(99, 110, 250)'),
        customdata=errors,
        hovertemplate='b = %{x:.2f}<br>phase = %{y:.2f} ± %{customdata:.2f} deg<extra></extra>',
    )
)
fig.update_layout(
    xaxis_title='imaginary part b',
    yaxis_title='phase / deg',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.05, y=0.95),
)
show_figure(fig, height=380)

## Good to know

The derivative of the arcsine is $1/\sqrt{1 - x^2}$, which diverges at $x = \pm 1$. The same uncertainty of
0.005 gives a much larger error close to the edge of the domain:

In [ ]:
for value in (0.5, 0.99):
    print(show(es.math.arcsin(Parameter(name='x', value=value, variance=0.005**2))))

Near $x = \pm 1$ the first-order variance becomes unreliable: the function is strongly curved there, and the
uncertainty may even reach beyond the domain.